<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.4-alloydb-bigquery/notebooks/GCP_Capstone_2.4_AlloyDB_BigQuery.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.4 AlloyDB pgvector & BigQuery Vector Search
**Netsetos GenAI Engineering — GCP Capstone**

SQL-native vector search, ScaNN vs HNSW, VECTOR_SEARCH(), and the 3-database decision framework.


## Setup
Note: AlloyDB requires a provisioned instance (minimum production config about $200/mo,
about Rs 17,000 at USD_INR = 85). This notebook demonstrates SQL patterns.
BigQuery examples use the free tier (1 TiB/month).


In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-bigquery "google-cloud-alloydb-connector[pg8000]" sqlalchemy
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS


## Cell 1: AlloyDB pgvector SQL Patterns
These SQL statements run on AlloyDB (not Colab). Study the patterns.


In [ ]:
# AlloyDB SQL patterns (reference — run on AlloyDB, not Colab)
ALLOYDB_SQL = """
-- 1. Enable extensions
CREATE EXTENSION IF NOT EXISTS vector;
CREATE EXTENSION IF NOT EXISTS google_ml_integration CASCADE;
CREATE EXTENSION IF NOT EXISTS alloydb_scann CASCADE;

-- 1b. Register the ONE model THIS store is embedded with. AlloyDB is its own store: the lane's
--     Firestore `chunks` is text-embedding-005, and a mirror of chunks here would register that instead.
--     Embeddings are regional-only, so the endpoint is us-central1 (never global).
--     Never register a second model against the same store - a store written by one
--     model and queried through another returns garbage, not an error.
--     Custom input transform: adds outputDimensionality: 768 to every Vertex request.
CREATE OR REPLACE FUNCTION documind_embed_in_768(model_id VARCHAR(100), input_text TEXT)
RETURNS JSON LANGUAGE sql AS $$
  SELECT json_build_object(
    'instances', json_build_array(json_build_object('content', input_text)),
    'parameters', json_build_object('outputDimensionality', 768))
$$;
CALL google_ml.create_model(
    model_id => 'documind-embed-768',
    model_provider => 'google',
    model_type => 'text_embedding',
    model_qualified_name => 'gemini-embedding-001',
    model_auth_type => 'alloydb_service_agent_iam',
    model_in_transform_fn => 'documind_embed_in_768',
    model_out_transform_fn => 'google_ml.vertexai_text_embedding_output_transform',
    model_request_url => 'https://us-central1-aiplatform.googleapis.com/v1/projects/'
        || 'documind-ai-YOUR-ID/locations/us-central1/publishers/google/'
        || 'models/gemini-embedding-001:predict'
);
-- The pre-registered id gemini-embedding-001 returns 3072-d; this registration wraps the same
-- model with a custom input transform that pins outputDimensionality: 768, so the column can be vector(768).

-- 2. Create table with embedding column
CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT,
    category VARCHAR(50),
    embedding vector(768)
);

-- 3. Create ScaNN index
CREATE INDEX idx_scann ON documents
USING scann (embedding cosine) WITH (num_leaves = 100);
ANALYZE documents;

-- 4. Vector search with cosine distance
SELECT id, content,
       embedding <=> '[0.1, 0.2, ...]'::vector AS distance
FROM documents
ORDER BY embedding <=> '[0.1, 0.2, ...]'::vector
LIMIT 5;

-- 5. Filtered search with JOIN
SELECT d.content, a.name, d.embedding <=> '[...]'::vector AS dist
FROM documents d JOIN authors a ON d.author_id = a.id
WHERE d.category = 'ai_ml'
ORDER BY d.embedding <=> '[...]'::vector LIMIT 5;

-- 6. Auto-embedding with generated column
-- documind-embed-768 is gemini-embedding-001 registered above with the 768-d input transform;
-- the pre-registered gemini-embedding-001 id returns 3072-d and the vector(768) column rejects it.
CREATE TABLE kb (
    id SERIAL PRIMARY KEY,
    content TEXT,
    embedding vector(768) GENERATED ALWAYS AS (
        embedding('documind-embed-768', content)
    ) STORED
);
"""
print(ALLOYDB_SQL)


## Cell 2: AlloyDB Python Connection Pattern


In [ ]:
# AlloyDB from Python: Language Connector + IAM auth (no password, no public IP).
import sqlalchemy
from google.cloud.alloydb.connector import Connector
from google import genai
from google.genai import types

INSTANCE_URI = (f"projects/{PROJECT_ID}/locations/us-central1"
                "/clusters/documind/instances/documind-primary")
# DB_USER must equal the identity behind your ADC: your own email in Colab after
# auth.authenticate_user(), sa-name@PROJECT.iam when running as a service account.
DB_USER = "you@example.com"   # the identity behind your ADC: your own e-mail in Colab (sa-name@PROJECT.iam as a service account)
RUN_ALLOYDB = False           # the lane has no AlloyDB: set True only after `gcloud services enable alloydb.googleapis.com`,
                              # a cluster + instance named as in INSTANCE_URI, and DB_USER created as an IAM database user


def make_engine():
    """AlloyDB is private-IP by default: Colab needs the connector (or the Auth Proxy)."""
    connector = Connector()

    def getconn():
        return connector.connect(INSTANCE_URI, "pg8000",
                                 user=DB_USER, db="documind", enable_iam_auth=True)

    return sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)


def embed(client, text, task_type):
    """gemini-embedding-001 takes ONE text per request on Vertex AI - loop, never a list."""
    return client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=768),
    ).embeddings[0].values


def search(query, k=5):
    # Embeddings are regional-only -> us-central1. Gemini 3.x generation would need a
    # SECOND client on location="global".
    ai = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")
    q = str(embed(ai, query, "RETRIEVAL_QUERY"))
    with make_engine().connect() as db:
        return db.execute(sqlalchemy.text(f"""
            SELECT content, embedding <=> CAST(:q AS vector) AS distance
            FROM documents
            ORDER BY embedding <=> CAST(:q AS vector) LIMIT {k}
        """), {"q": q}).fetchall()


if PROJECT_ID == 'documind-ai-YOUR-ID' or not RUN_ALLOYDB:   # skipped on the lane: no cluster exists
    print('Set PROJECT_ID and provision an AlloyDB cluster, then call search("...").')
    print(f'Connector target: {INSTANCE_URI}')
    print(f'IAM principal:    {DB_USER} (enable_iam_auth=True, nothing to paste)')
else:
    for content, dist in search('How fast is AlloyDB for vector queries?'):
        print(f'  [{1 - dist:.4f}] {content}')


## Cell 3: BigQuery VECTOR_SEARCH (Reference - two prerequisites)


In [ ]:
from google.cloud import bigquery

# Two prerequisites before ANY of this SQL runs:
#   1. A BigQuery -> Vertex AI connection (the model call has no other way out):
#        bq mk --connection --location=us --connection_type=CLOUD_RESOURCE vertex_conn
#      then grant that connection's service account roles/aiplatform.user.
#   2. A base table of at least 10 MB - below that CREATE VECTOR INDEX succeeds but
#      the index is never populated (coverage 0%) and VECTOR_SEARCH runs brute force;
#      the 3-row demo table below is far under it.
DATASET = 'documind_vectors'
CONNECTION = f'{PROJECT_ID}.us.vertex_conn'

if PROJECT_ID != 'documind-ai-YOUR-ID':
    bq = bigquery.Client(project=PROJECT_ID)
    bq.create_dataset(f'{PROJECT_ID}.{DATASET}', exists_ok=True)
    print(f'Dataset {DATASET} ready.')
else:
    print('Set PROJECT_ID to create the dataset; the SQL below is reference only.')

# BigQuery SQL patterns
BQ_SQL = f"""
-- 1. Create remote embedding model: the SAME model the store is written with.
--    gemini-embedding-001 returns 3072-d by default; request 768-d output (MRL
--    truncation) or BigQuery stores 3072-d vectors that no longer match the 768-d
--    store contract.
CREATE OR REPLACE MODEL `{DATASET}.embed_model`
  REMOTE WITH CONNECTION `{CONNECTION}`
  OPTIONS (ENDPOINT = 'gemini-embedding-001');

-- 2. Generate embeddings. The STRUCT pins the output to 768 dimensions, which is
--    what the store's contract says; without it BigQuery stores 3072-d vectors.
CREATE OR REPLACE TABLE `{DATASET}.doc_embeddings` AS
SELECT * FROM AI.GENERATE_EMBEDDING(
  MODEL `{DATASET}.embed_model`,
  (SELECT 'doc_1' AS id, 'Transformers use self-attention' AS content
   UNION ALL SELECT 'doc_2', 'RAG combines retrieval with generation'
   UNION ALL SELECT 'doc_3', 'AlloyDB is a managed PostgreSQL database'),
  STRUCT(768 AS output_dimensionality, 'RETRIEVAL_DOCUMENT' AS task_type,
         TRUE AS flatten_json_output)
);

-- 3. Create vector index. Needs a base table of at least 10 MB - below that the
--    statement succeeds but the index is never populated (coverage 0%); the 3-row
--    table above is far under it. Vector indexes are not available in the Standard
--    edition (Enterprise or higher); verify on the BigQuery pricing page.
CREATE OR REPLACE VECTOR INDEX doc_idx
ON `{DATASET}.doc_embeddings`(ml_generate_embedding_result)
OPTIONS (index_type = 'IVF', distance_type = 'COSINE');

-- 4. Search (works without an index at this size - brute force)
SELECT base.content, distance
FROM VECTOR_SEARCH(
  TABLE `{DATASET}.doc_embeddings`,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result AS embedding
   FROM AI.GENERATE_EMBEDDING(
     MODEL `{DATASET}.embed_model`,
     (SELECT 'How does attention work?' AS content),
     STRUCT(768 AS output_dimensionality, 'RETRIEVAL_QUERY' AS task_type,
            TRUE AS flatten_json_output))),
  top_k => 3, distance_type => 'COSINE'
);
"""
print(BQ_SQL)


## Cell 4: Decision Framework Calculator


In [ ]:
USD_INR = 85  # rate as of 2026-09-03
ALLOYDB_USD = 200        # minimum production config: 2 vCPU / 16 GiB, no read pool
FIRESTORE_100K_USD = 15  # storage + reads for the DocuMind workload at 100K docs


def recommend_vector_db(num_docs, need_realtime=True, need_sql=False, budget_monthly=0):
    print(f'\nDocuMind Vector DB Recommendation:')
    print(f'  Documents: {num_docs:,}')
    print(f'  Real-time: {need_realtime}')
    print(f'  SQL needed: {need_sql}')
    print(f'  Budget: Rs {budget_monthly:,}/mo\n')

    if num_docs < 10000 and not need_sql:
        print('  Firestore (free tier)')
        print('  - find_nearest() with cosine distance')
        print('  - Flat index (exact KNN)')
        print('  - Cost: Rs 0/mo')
    elif need_realtime and (need_sql or num_docs >= 10000):
        print('  AlloyDB with ScaNN')
        print('  - pgvector <=> operator')
        print('  - ScaNN index (1-30ms latency)')
        print(f'  - Cost: about ${ALLOYDB_USD}/mo = Rs {ALLOYDB_USD * USD_INR:,}/mo '
              '(minimum config, storage + serving)')
    else:
        print('  BigQuery VECTOR_SEARCH')
        print('  - AI.GENERATE_EMBEDDING() + VECTOR_SEARCH()')
        print('  - IVF/TreeAH index')
        print('  - Cost: Rs 0-850/mo for 10K-1M docs = storage + a small query')
        print('    allowance. Queries are billed separately as on-demand data scanned,')
        print('    so the two figures have different bases and must not be added.')

# Test scenarios
recommend_vector_db(500, need_realtime=True, need_sql=False)
recommend_vector_db(100000, need_realtime=True, need_sql=True)
recommend_vector_db(5000000, need_realtime=False, need_sql=False)

# One AlloyDB figure and one Firestore figure for the whole module.
print(f'\nFirestore at 100,000 DocuMind docs: about ${FIRESTORE_100K_USD}/mo = '
      f'Rs {FIRESTORE_100K_USD * USD_INR:,}/mo (storage plus reads).')
print('Storage is never the driver: 10,000 documents with 768-d embeddings are')
print('only about 80 MB (10K x 8 KB).')


## Cell 5: Compare Index Types


In [ ]:
index_comparison = {
    'IVFFlat': {'algorithm': 'K-means', 'build': 'Fast', 'memory': 'Medium',
                'in_memory_latency': 'Good', 'out_of_memory': 'OK', 'availability': 'Any PostgreSQL'},
    'HNSW': {'algorithm': 'Graph', 'build': 'Slow', 'memory': 'Largest',
             'in_memory_latency': 'Great', 'out_of_memory': 'Degrades badly', 'availability': 'Any PostgreSQL'},
    'ScaNN': {'algorithm': 'Tree+quantization', 'build': '10x faster', 'memory': '4x smaller',
              'in_memory_latency': 'Up to 4x better', 'out_of_memory': '10x better', 'availability': 'AlloyDB only'},
    'IVF (BQ)': {'algorithm': 'K-means', 'build': 'Auto', 'memory': 'Serverless',
                  'in_memory_latency': 'Seconds', 'out_of_memory': 'N/A', 'availability': 'BigQuery'},
    'TreeAH (BQ)': {'algorithm': 'Tree+hashing', 'build': '10x faster', 'memory': 'Serverless',
                     'in_memory_latency': 'Batch optimized', 'out_of_memory': 'N/A', 'availability': 'BigQuery'},
}

print(f"{'Index':<15} {'Algorithm':<20} {'Build':<12} {'Memory':<12} {'Availability':<18}")
print('-' * 80)
for name, props in index_comparison.items():
    print(f"{name:<15} {props['algorithm']:<20} {props['build']:<12} {props['memory']:<12} {props['availability']:<18}")


## ✅ Module 2 Complete!

- ✅ 2.1: Token Economics — tokenization, multilingual costs, context budgeting
- ✅ 2.2: Embeddings — gemini-embedding-001, cosine similarity, task types
- ✅ 2.3: Firestore Vector Search — find_nearest(), complete RAG pipeline
- ✅ 2.4: AlloyDB & BigQuery — SQL vectors, ScaNN/HNSW, decision framework

**Next: Module 3 — Prompt Engineering & Structured Output**
